In [25]:
from google import genai
from IPython.display import display, Markdown
from dotenv import load_dotenv

load_dotenv()
client =genai.Client()

interaction = client.interactions.create(
    model= "gemini-3.7-flash",
    input="請逐步教學含視窗化操作指令, 如何使用raspberry樹莓派架設出家用的NAS伺服器?")
display(Markdown(interaction.output_text))
      

使用樹莓派（Raspberry Pi）架設家用 NAS，最推薦且具備**強大網頁視窗化管理介面（Web GUI）**的系統是 **OpenMediaVault (簡稱 OMV)**。

以下為你整理從零開始的完整逐步教學，除了初期安裝需要貼上一行指令外，其餘操作皆為**視窗圖形化介面**。

---

### 必備硬體準備
1. **樹莓派**：建議 Raspberry Pi 4 或 5（具備 USB 3.0 與 Gigabit 網路孔，傳輸速度才夠快）。
2. **MicroSD 卡**：16GB 或以上（安裝系統用）。
3. **外接硬碟 / SSD**：儲存資料用（建議有獨立供電，或使用外接硬碟外接盒）。
4. **網路線**：建議將樹莓派直接插在路由器（Router）上以確保傳輸穩定。
5. **電腦一台**：用來設定與操作視窗介面。

---

### 第一階段：製作系統開機卡（視窗化工具）

我們使用官方的 **Raspberry Pi Imager** 視窗軟體來製作開機卡。

1. **下載軟體**：至樹莓派官網下載並安裝 [Raspberry Pi Imager](https://www.raspberrypi.com/software/)。
2. **插入 MicroSD 卡**至電腦。
3. **開啟 Raspberry Pi Imager**：
   * **裝置 (Device)**：選擇你的樹莓派型號（例如 Raspberry Pi 4）。
   * **作業系統 (OS)**：點選 `Raspberry Pi OS (other)` $\to$ 選擇 **`Raspberry Pi OS Lite (64-bit)`**（Lite 版本最輕量，無內建桌面，效能最好）。
   * **儲存空間 (Storage)**：選擇你的 MicroSD 卡。
4. **點擊「下一步 (Next)」並編輯設定（齒輪圖示/客製化設定）**：
   * 設定主機名稱（例如：`raspberrypi`）。
   * 設定使用者名稱與密碼（例如帳號：`pi`，密碼自己設定並記住）。
   * **務必勾選「開啟 SSH」**，選擇「使用密碼驗證」。
   * 設定時區為 `Asia/Taipei`。
5. **點擊「儲存」並開始「寫入」**，等待完成後將 MicroSD 卡拔出，插入樹莓派。

---

### 第二階段：一鍵安裝 OpenMediaVault (OMV)

將樹莓派接上電源與網路線開機，等待約 2 分鐘。

1. 在電腦上打開終端機（Windows 請開 **PowerShell**，Mac 請開 **Terminal**）。
2. 輸入以下指令連線至樹莓派（將 `pi` 換成你剛剛設定的帳號）：
   ```bash
   ssh pi@raspberrypi.local
   ```
   *(若連不上，請至路由器後台查看樹莓派的 IP，改輸入 `ssh pi@你的樹莓派IP`，輸入密碼登入)*
3. **貼上 OMV 一鍵安裝指令**並按下 Enter（這是**唯一**需要打指令的步驟）：
   ```bash
   wget -O - https://github.com/OpenMediaVault-Plugin-Developers/installScript/raw/master/install | sudo bash
   ```
4. 系統會自動下載並安裝（耗時約 15~30 分鐘，視網路速度而定）。安裝完成後，樹莓派會自動重開機。

---

### 第三階段：進入視窗化介面設定 NAS (OMV Web GUI)

接下來**所有操作都在瀏覽器中完成**！

#### 1. 登入 OMV 管理後台
* 打開電腦瀏覽器，輸入樹莓派的 IP 位址（例如：`http://192.168.1.100` 或 `http://raspberrypi.local`）。
* **預設登入帳號**：`admin`
* **預設登入密碼**：`openmediavault`
*(登入後可至右上角齒輪變更為繁體中文介面，並強烈建議修改預設密碼)*

---

#### 2. 初始化與掛載外接硬碟
將外接硬碟插上樹莓派的 **藍色 USB 3.0 埠**。

1. **格式化硬碟（若為新硬碟或硬碟內資料不需要）**：
   * 點選左側選單 **「儲存區」 $\to$ 「磁碟 (Disks)」**，確認有看到你的外接硬碟。
   * 點選左側 **「儲存區」 $\to$ 「檔案系統 (File Systems)」**。
   * 點擊上方 **「＋（建立）」**，選擇你的硬碟，檔案系統建議選擇 `EXT4` 或 `BTRFS`，點擊儲存並確認格式化。
2. **掛載硬碟**：
   * 在「檔案系統」頁面，點擊上方 **「＋（掛載）」** 圖示。
   * 選擇剛才建立的檔案系統，點擊「儲存」。
   * **重要**：畫面上方出現**黃色提示條**時，務必點擊**打勾「套用變更」**。

---

#### 3. 建立共用資料夾 (Shared Folders)
1. 點選左側 **「儲存區」 $\to$ 「共享資料夾 (Shared Folders)」**。
2. 點擊上方 **「＋」號**：
   * **名稱**：輸入資料夾名稱（例如：`Family_NAS`）。
   * **檔案系統**：選擇剛才掛載的外接硬碟。
   * **權限**：預設「系統管理者：讀/寫，使用者：讀/寫，其他人：無存取權限」。
3. 點擊「儲存」，並在黃色提示條點擊「套用變更」。

---

#### 4. 建立使用者帳號 (Users)
1. 點選左側 **「使用者」 $\to$ 「使用者 (Users)」**。
2. 點擊 **「＋」號**：
   * **名稱**：輸入你要用來連線的使用者名稱（例如：`user1`）。
   * **密碼**：輸入該使用者的密碼。
3. 點擊「儲存」，並點擊黃色提示條「套用變更」。
4. 點擊剛建立的使用者，選擇上方的 **「權限 (Permissions)」** 圖示，確保對剛才建立的 `Family_NAS` 資料夾勾選 **「讀寫 (Read/Write)」**。

---

#### 5. 開啟 Windows/Mac 檔案分享 (SMB/CIFS)
1. 點選左側 **「服務」 $\to$ 「SMB/CIFS」 $\to$ 「設定」**：
   * 將 **「已啟用 (Enabled)」** 開關打開。
   * 點擊「儲存」。
2. 切換到上方頁籤 **「共享 (Shares)」**：
   * 點擊 **「＋」號**。
   * **共用資料夾**：選擇剛才建立的 `Family_NAS`。
   * 將 **「已啟用 (Enabled)」** 打開。
   * 點擊「儲存」，最後在黃色提示條點擊「套用變更」。

---

### 第四階段：從電腦連線至 NAS

現在你的家用 NAS 已經架設完成！

#### Windows 電腦連線：
1. 打開「檔案總管」，在上方網址列輸入：
   `\\樹莓派IP` （例如：`\\192.168.1.100`）並按 Enter。
2. 輸入剛剛在 OMV 建立的 **使用者名稱與密碼**。
3. 你就會看到 `Family_NAS` 資料夾，可以按右鍵選擇「連線網路磁碟機」，將它變成電腦裡的 `Z:` 槽！

#### Mac 電腦連線：
1. 打開 Finder，按下鍵盤快速鍵 `Command + K`。
2. 伺服器位址輸入：`smb://樹莓派IP` （例如：`smb://192.168.1.100`）並點擊連線。
3. 選擇「註冊使用者」，輸入帳號密碼即可連入。

#### 手機連線：
* **iOS**：可使用內建「檔案」App $\to$ 點右上角「...」 $\to$ 「連接伺服器」 $\to$ 輸入 `smb://樹莓派IP`。
* **Android**：可下載「Owlfiles」或「Cx 檔案總管」，新增 SMB 連線即可。